# 🇮🇳 TravelSense AI – Data Collection, Preprocessing & EDA Notebook

> **⚠️ DISCLAIMER:** This notebook utilizes a synthetic prototype dataset covering 200 Indian travel destinations. All monetary values are represented in **Indian Rupees (INR / ₹)** and are synthetically generated for machine learning and data science pipeline demonstration purposes. They do not represent real-time market prices.

## 📌 Objectives
1. Inspect raw Indian destination travel dataset (`data/indian_destinations_raw.csv`).
2. Identify missing values and duplicate records.
3. Perform robust data cleaning (Imputation, Duplicate Removal, One-Hot Encoding, Feature Scaling).
4. Compare dataset stats **Before vs. After Cleaning**.
5. Ensure 100% pipeline reproducibility with a fixed random seed (`SEED = 42`).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure reproducibility
SEED = 42
np.random.seed(SEED)

# Add src to path
sys.path.append(os.path.abspath('../src'))
from preprocess import preprocess_travel_data, load_raw_dataset, inspect_dataset

# Visual styles
sns.set_theme(style="whitegrid", palette="Blues_r")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Data Inspection BEFORE Cleaning

In [ ]:
# Load raw dataset
df_raw = load_raw_dataset('../data/indian_destinations_raw.csv')
stats_raw = inspect_dataset(df_raw)

print("=== RAW DATASET SUMMARY ===")
print(f"Total Rows: {stats_raw['total_rows']}")
print(f"Total Columns: {stats_raw['total_columns']}")
print(f"Duplicate Rows Found: {stats_raw['duplicate_count']}")
print("\nMissing Values per Column:")
for col, count in stats_raw['missing_values'].items():
    if count > 0:
        print(f"  - {col}: {count} missing values")

df_raw.head()

### Visualizing Missing Values in Raw Dataset

In [ ]:
plt.figure(figsize=(8, 4))
missing_counts = df_raw.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]
sns.barplot(x=missing_counts.index, y=missing_counts.values, palette="Reds_r")
plt.title("Missing Values Count per Feature (Before Cleaning)", fontweight='bold')
plt.ylabel("Count")
plt.xlabel("Features")
plt.show()

## 2. Execute Data Preprocessing & Cleaning Pipeline

In [ ]:
# Execute modular preprocessing function from src/preprocess.py
pipe_results = preprocess_travel_data(file_path='../data/indian_destinations_raw.csv', save_cleaned=True, seed=SEED)

df_cleaned = pipe_results['df_cleaned']
df_preprocessed = pipe_results['df_preprocessed']
stats_cleaned = pipe_results['stats_after']

print("=== CLEANED DATASET SUMMARY ===")
print(f"Total Rows (After Deduplication): {stats_cleaned['total_rows']}")
print(f"Duplicate Rows Remaining: {stats_cleaned['duplicate_count']}")
print(f"Total Missing Values: {sum(stats_cleaned['missing_values'].values())}")

## 3. Data Inspection AFTER Cleaning & Imputation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before vs After Rows & Duplicates
categories = ['Raw Rows', 'Cleaned Rows', 'Duplicates Raw', 'Duplicates Cleaned']
values = [stats_raw['total_rows'], stats_cleaned['total_rows'], stats_raw['duplicate_count'], stats_cleaned['duplicate_count']]

sns.barplot(x=categories, y=values, ax=axes[0], palette="Blues_d")
axes[0].set_title("Dataset Record Counts Before vs After Cleaning", fontweight='bold')
axes[0].set_ylabel("Number of Records")

# Total Daily Cost Distribution in INR (₹)
sns.histplot(df_cleaned['total_daily_cost_inr'], kde=True, ax=axes[1], color="#0284C7")
axes[1].set_title("Distribution of Total Daily Cost in INR (₹)", fontweight='bold')
axes[1].set_xlabel("Daily Cost (INR ₹)")

plt.tight_layout()
plt.show()

## 4. Categorical Encoding & Scaled Feature Matrix Overview

In [ ]:
print(f"Preprocessed ML Matrix Shape (One-Hot Encoded + Scaled): {df_preprocessed.shape}")
df_preprocessed.head(3)

## 5. Feature Correlation Heatmap (Cleaned INR Cost Components)

In [ ]:
plt.figure(figsize=(10, 6))
cost_cols = [
    'average_hotel_cost', 'average_food_cost',
    'average_local_transport_cost', 'average_activity_cost',
    'total_daily_cost_inr', 'popularity_score', 'recommended_duration'
]
corr_matrix = df_cleaned[cost_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap="YlGnBu", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Indian Destination Costs (INR ₹)", fontweight='bold')
plt.tight_layout()
plt.show()